In [0]:
from pyspark.sql.functions import current_timestamp, lit, to_date, col
from pyspark.sql.types import *

storage_account = "retailprojec1data"
container = "bronze"

source_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/inventory/raw/"
checkpoint_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/inventory/_checkpoints/"
target_table_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/inventory_delta/"
schema_location = f"abfss://{container}@{storage_account}.dfs.core.windows.net/inventory/_schema/"

inventory_schema = StructType([
    StructField("snapshot_date", StringType(), True), 
    StructField("store_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("stock_on_hand", IntegerType(), True),
    StructField("stock_in_transit", IntegerType(), True),
    StructField("stock_on_order", IntegerType(), True),
    StructField("reorder_point", IntegerType(), True),
    StructField("reorder_quantity", IntegerType(), True),
    StructField("last_restock_date", StringType(), True),  
    StructField("warehouse_location", StringType(), True),
     StructField("year", IntegerType(), True),    
    StructField("month", IntegerType(), True),    
    StructField("day", IntegerType(), True),      
])

In [0]:
from pyspark.sql.functions import current_timestamp, lit, to_date, coalesce, col
from pyspark.sql.types import *

def parse_date_flexible(col_obj):
    """Try common date formats; returns NULL if none parse successfully."""
    return coalesce(
        to_date(col_obj, "yyyy-MM-dd"),
        to_date(col_obj, "M/d/yyyy"),
        to_date(col_obj, "d/M/yyyy"),
        to_date(col_obj, "yyyy/MM/dd"),
        to_date(col_obj, "dd-MM-yyyy"),
    )

In [0]:
inventory_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.partitionColumns", "year,month,day")
    .option("header", "true")
    .schema(inventory_schema)
    .load(source_path))

In [0]:
# Add lineage columns and cast types
inventory_with_metadata = (inventory_stream
    .withColumn("snapshot_date", parse_date_flexible(col("snapshot_date")))
    .withColumn("last_restock_date", parse_date_flexible(col("last_restock_date")))
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_system", lit("inventory_csv_drops"))
    .withColumn("source_file", col("_metadata.file_path")))

In [0]:
# Write stream into Bronze Delta table
query = (inventory_with_metadata.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)  # Process all available files, then stop
    .start(target_table_path))
print("✅ Inventory ingestion complete")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS retail.bronze_inventory
USING DELTA
LOCATION '{target_table_path}'
""")

In [0]:
%sql
select * from retail.bronze_inventory;